# تدريب المُشكِّل (قابل للاستئناف)
**قبل البدء:** وقت التشغيل ← تغيير نوع وقت التشغيل ← **T4 GPU** ← حفظ.
شغّل الخلايا **واحدة واحدة** وانتظر `تم ✔` أو ✓. النقاط تُحفظ في Google Drive (مجلد `dia`)، فإن انقطعت الجلسة فأعد تشغيل الخلايا ١ إلى ٣ ثم الرابعة وستكمل من آخر نقطة.
التدريب على أجزاء (٤٠ دقيقة للجزء). كرّر الخلية ٤ مرتين أو ثلاثًا. اترك الشاشة مضاءة والتبويب مفتوحًا أثناء الجزء.

In [ ]:
# ١) ربط Google Drive لحفظ النقاط (ستظهر نافذة موافقة، اقبلها)
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/dia', exist_ok=True)
print('تم ✔ Drive')

In [ ]:
%%bash
set -e
cd /content
rm -rf dhad Open-Hadith-Data arabic-text-diacritization
git clone -q --depth 1 https://github.com/kaledgmal71-commits/dhad.git
git clone -q --depth 1 https://github.com/mhashim6/Open-Hadith-Data.git
git clone -q --depth 1 https://github.com/AliOsm/arabic-text-diacritization.git
test -f dhad/tools/dia_train.py && test -f dhad/tools/dia_eval.py && test -f arabic-text-diacritization/dataset/test.txt
nvidia-smi -L || echo "تنبيه: لا توجد بطاقة رسوم، فعّل T4 GPU"
python3 -c "import cupy" 2>/dev/null || pip -q install cupy-cuda12x
python3 dhad/tools/diacritizer_data.py Open-Hadith-Data hadith.npz
test -f hadith.npz
echo "تم ✔ التنزيل والبيانات"

In [ ]:
%%bash
set -e
cd /content
(cd dhad && git pull -q) || true
D=/content/drive/MyDrive/dia
RES=""
if [ -f $D/dia_big.json ] && [ -f $D/dia_big.npz ]; then RES="--resume $D/dia_big"; echo "استئناف من آخر نقطة"; else echo "بداية جديدة"; fi
python3 dhad/tools/dia_train.py hadith.npz $D/dia_big --minutes 40 --d 384 --layers 6 --heads 6 --ff 1536 --T 256 --batch 64 --lr 5e-4 --mask-final-none --val arabic-text-diacritization/dataset/val.txt $RES
test -f $D/dia_big.npz
echo "تم ✔ جزء من التدريب (كرّر هذه الخلية لجزء آخر)"

In [ ]:
%%bash
set -e
cd /content
tail -3 /content/drive/MyDrive/dia/dia_big.log

In [ ]:
%%bash
set -e
cd /content
python3 dhad/tools/dia_eval.py /content/drive/MyDrive/dia/dia_big arabic-text-diacritization/dataset/test.txt
echo "تم ✔ القياس النهائي (مرة واحدة فقط، بعد انتهاء كل الأجزاء)"